# sentiment Analysis using RNN

In [52]:
import numpy as np
import torch

## get data

In [53]:
reviews = [
    "I love this product it's amazing!",
    "Excellent service highly recommended!",
    "The best experience ever!",
    "Great quality and value for money.",
    "Absolutely fantastic would buy again!",
    "Superb performance definitely worth it.",
    "Loved every moment of using this!",
    "Outstanding customer support, thank you!",
    "A must-have item highly satisfied!",
    "Perfectly meets all my expectations!",

    "This product is terrible don't buy it!",
    "Disappointing experience not worth the money.",
    "The worst service I've ever encountered.",
    "Poor quality completely broken.",
    "Not worth the time or money.",
    "Horrible customer support very frustrating.",
    "I regret purchasing this item.",
    "Badly designed doesn't work as intended.",
    "Extremely dissatisfied with the product.",
    "Complete waste of money avoid at all costs."
]

# class labels
labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## Build veocabulary

In [54]:
# every word in voccab 
MAX_LENGTH =  10

# create empy vocab
vocabulary = {
    # keep tokens of fixed length
    # if no. of token is smaller then max_len we add padding
    "<PAD>" : 0
}


In [55]:
# split all reviews in tokens (word)
for review in reviews:
    for word in review.split():

        # check if word already present in vocab
        if word not in vocabulary:
            vocabulary[word] = len(vocabulary)

print(vocabulary)

{'<PAD>': 0, 'I': 1, 'love': 2, 'this': 3, 'product': 4, "it's": 5, 'amazing!': 6, 'Excellent': 7, 'service': 8, 'highly': 9, 'recommended!': 10, 'The': 11, 'best': 12, 'experience': 13, 'ever!': 14, 'Great': 15, 'quality': 16, 'and': 17, 'value': 18, 'for': 19, 'money.': 20, 'Absolutely': 21, 'fantastic': 22, 'would': 23, 'buy': 24, 'again!': 25, 'Superb': 26, 'performance': 27, 'definitely': 28, 'worth': 29, 'it.': 30, 'Loved': 31, 'every': 32, 'moment': 33, 'of': 34, 'using': 35, 'this!': 36, 'Outstanding': 37, 'customer': 38, 'support,': 39, 'thank': 40, 'you!': 41, 'A': 42, 'must-have': 43, 'item': 44, 'satisfied!': 45, 'Perfectly': 46, 'meets': 47, 'all': 48, 'my': 49, 'expectations!': 50, 'This': 51, 'is': 52, 'terrible': 53, "don't": 54, 'it!': 55, 'Disappointing': 56, 'not': 57, 'the': 58, 'worst': 59, "I've": 60, 'ever': 61, 'encountered.': 62, 'Poor': 63, 'completely': 64, 'broken.': 65, 'Not': 66, 'time': 67, 'or': 68, 'Horrible': 69, 'support': 70, 'very': 71, 'frustrating

## Encode all reviews 

In [56]:
def encode(review):

    #split review in tokens
    tokens = [vocabulary[word] for word in review.split()]

    # add padding if required
    while len(tokens) <MAX_LENGTH:
        tokens.append(0)

    return tokens

In [57]:
encoded_reviews = [encode(review) for review in reviews] 
print(encoded_reviews)

[[1, 2, 3, 4, 5, 6, 0, 0, 0, 0], [7, 8, 9, 10, 0, 0, 0, 0, 0, 0], [11, 12, 13, 14, 0, 0, 0, 0, 0, 0], [15, 16, 17, 18, 19, 20, 0, 0, 0, 0], [21, 22, 23, 24, 25, 0, 0, 0, 0, 0], [26, 27, 28, 29, 30, 0, 0, 0, 0, 0], [31, 32, 33, 34, 35, 36, 0, 0, 0, 0], [37, 38, 39, 40, 41, 0, 0, 0, 0, 0], [42, 43, 44, 9, 45, 0, 0, 0, 0, 0], [46, 47, 48, 49, 50, 0, 0, 0, 0, 0], [51, 4, 52, 53, 54, 24, 55, 0, 0, 0], [56, 13, 57, 29, 58, 20, 0, 0, 0, 0], [11, 59, 8, 60, 61, 62, 0, 0, 0, 0], [63, 16, 64, 65, 0, 0, 0, 0, 0, 0], [66, 29, 58, 67, 68, 20, 0, 0, 0, 0], [69, 38, 70, 71, 72, 0, 0, 0, 0, 0], [1, 73, 74, 3, 75, 0, 0, 0, 0, 0], [76, 77, 78, 79, 80, 81, 0, 0, 0, 0], [82, 83, 84, 58, 85, 0, 0, 0, 0, 0], [86, 87, 34, 88, 89, 90, 48, 91, 0, 0]]


## define  a dataset

In [58]:
from torch.utils.data import Dataset

# create custome dataset to have review in tensor format
class SentimentDataset(Dataset):
 
    def __init__(self):
        # super.__init__(self)
        # conveert and store the encoded reviews in tensor
        # we converted to long buz if use fload decoder part will not work here
        self.x = torch.tensor(encoded_reviews,dtype=torch.long)

        # convert and store corresponding lables to torch
        self.y = torch.tensor(labels,dtype=torch.float32)

    # return len of dataset
    def __len__(self):
        return len(self.x)

    # retunr value at specified index pos
    def __getitem__(self, index):
        return self.x[index], self.y[index]

In [59]:
# creating instances of Sen
dataset = SentimentDataset()

## create DataLoader

In [60]:
from torch.utils.data import DataLoader

# create a data loader to load dataset
loader = DataLoader(dataset=dataset,shuffle=True,batch_size=2)

## create model

In [61]:
# create sub class of torch.nn.Module to represent model
class SentimentAnalysis(torch.nn.Module):

    # create initilaizer
    def __init__(self):
        # initilaize parent
        super().__init__()

        # add required layers
        self.embedding = torch.nn.Embedding(
            num_embeddings=len(vocabulary),
            embedding_dim=8,        # hyperpearameter
        )

        # add rnn layer
        self.rnn = torch.nn.RNN(
            input_size=8,
            hidden_size=16,     # hyper parameter
            batch_first=True
        )

        # as it is Binary classifiacation add linear layer and sigmoid 
        self.linear = torch.nn.Linear(16,1)

        #apply sigmoid activation function too get final result as 0 or 1
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self,x):

        # convert input embedding
        x = self.embedding(x)

        # pass thus embeddings to rnn layer
        # hidden state is in shape of (1,batch_size, hidden_state)
        output, hidden = self.rnn(x)

        # convert shape of hidden state : (batch_size, hidden_size)
        hidden = hidden.squeeze(0)

        # pass reshaped hidden state to linear layer
        x = self.linear(hidden)

        # get the binary classification result  using sigmoid
        x = self.sigmoid(x)

        # return thee final result to caller
        return x

## detecct GPU



In [62]:
device= ""

if torch.cuda.is_available():
    device = 'cuda'
elif torch.mps.is_available():
    device='mps'
else :
    device='cpu'

print(device)


cuda


## Hyperparameters

In [63]:
# create instance
model = SentimentAnalysis()

# move model to device
model = model.to(device)

### define hyperparameters

In [64]:
# define loss function
losss_funtion = torch.nn.BCELoss()

# no of epoch
epoch = 50

# learning  rate 
learning_rate = 0.01

# optimizer
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)


## Traning loop

In [65]:
for epoch in range(epoch):

    # totla loss
    total_loss = 0


    # iterate over the data loader to get data in a required batch_size
    for x,y in loader:

        # move x,y to device
        # as loss function is BCELoss convert y.view(-1,1)
        x = x.to(device)
        y = y.to(device).view(-1,1)

        # pass input model get predictions
        # forward method of model get invoke  
        predictions = model(x)

         # calculate loss
        loss = losss_funtion(predictions,y)

        # zzero out gradient
        optimizer.zero_grad()

        # calculate gradients
        loss.backward()

        #optimze model parameters
        optimizer.step()

        # collection loss
        total_loss += loss.item()

    print(f"epoch = {epoch} , total loss = {total_loss}")

epoch = 0 , total loss = 7.8770060539245605
epoch = 1 , total loss = 7.314712762832642
epoch = 2 , total loss = 6.972055554389954
epoch = 3 , total loss = 6.9908610582351685
epoch = 4 , total loss = 6.823237180709839
epoch = 5 , total loss = 6.4271950125694275
epoch = 6 , total loss = 5.552172094583511
epoch = 7 , total loss = 4.574747562408447
epoch = 8 , total loss = 2.687554880976677
epoch = 9 , total loss = 1.4332863315939903
epoch = 10 , total loss = 0.6243293099105358
epoch = 11 , total loss = 0.3560394747182727
epoch = 12 , total loss = 0.22025751788169146
epoch = 13 , total loss = 0.1603372059762478
epoch = 14 , total loss = 0.12546776793897152
epoch = 15 , total loss = 0.10298690386116505
epoch = 16 , total loss = 0.08474724134430289
epoch = 17 , total loss = 0.07292515970766544
epoch = 18 , total loss = 0.06478840531781316
epoch = 19 , total loss = 0.05742822587490082
epoch = 20 , total loss = 0.05151832848787308
epoch = 21 , total loss = 0.046669891104102135
epoch = 22 , tot

## test model by passing unseen data

In [66]:
def predict(review):
    # change the mode of model to eval
    model.eval()

    # encode reviewe
    encoded_review = torch.tensor([encode(review)],dtype = torch.long).to(device)

    # disable learning 
    with torch.no_grad():
        # predict 
        prediction = model(encoded_review)

        print(f"prediction = {prediction}")

        if prediction > 0.5:
            print("review is positive")
        else:
            print(f"review is negative")


In [69]:
predict("highly satisfied!")

prediction = tensor([[0.8627]], device='cuda:0')
review is positive
